# SMM4H-HeaRD 2026 – Task 2: Insomnia Detection
## Subtask 1: Binary Classification using `google/gemma-3-1b-it` + QLoRA

**Model:** `google/gemma-3-1b-it`  
**GPU:** T4 16GB  
**Strategy:** QLoRA fine-tuning + keyword hybrid for F1 > 0.90

## Step 0 – Install Dependencies

In [1]:
!pip install -q transformers peft bitsandbytes accelerate datasets scikit-learn huggingface_hub
print('Installation complete!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.6 MB/s eta 0:00:00
Installation complete!


## Step 1 – Set Environment & Check GPU

In [2]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Step 2 – Upload Files

In [3]:
from google.colab import files
print('Upload: training_corpus.csv, validation_corpus.csv, test_corpus.csv,')
print('        subtask_1_train.json, subtask_1_valid.json')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

Upload: training_corpus.csv, validation_corpus.csv, test_corpus.csv,
        subtask_1_train.json, subtask_1_valid.json


Saving subtask_1_train.json to subtask_1_train.json
Saving subtask_1_valid.json to subtask_1_valid.json
Saving test_corpus.csv to test_corpus.csv
Saving training_corpus.csv to training_corpus.csv
Saving validation_corpus.csv to validation_corpus.csv
Uploaded: ['subtask_1_train.json', 'subtask_1_valid.json', 'test_corpus.csv', 'training_corpus.csv', 'validation_corpus.csv']


## Step 3 – HuggingFace Login

In [17]:
# # Accept Gemma license at https://huggingface.co/google/gemma-3-1b-it first
# from huggingface_hub import notebook_login
# notebook_login()

from huggingface_hub import login
from google.colab import userdata

# Automatically retrieves the token from Colab Secrets
token = userdata.get('HF_TOKEN')
login(token)


## Step 4 – Load & Prepare Data

In [4]:
import pandas as pd
import json
from collections import Counter

# Load CSVs
train_df = pd.read_csv('training_corpus.csv')
val_df   = pd.read_csv('validation_corpus.csv')
test_df  = pd.read_csv('test_corpus.csv')

# Load JSON labels
with open('subtask_1_train.json') as f:
    train_labels = json.load(f)
with open('subtask_1_valid.json') as f:
    val_labels = json.load(f)

# Merge labels
def merge_labels(df, label_dict):
    df = df.copy()
    df['note_id_str'] = df['note_id'].astype(str)
    df['label'] = df['note_id_str'].map(
        lambda nid: label_dict.get(nid, {}).get('Insomnia', None)
    )
    df = df.dropna(subset=['label'])
    df['label_int'] = (df['label'] == 'yes').astype(int)
    return df.reset_index(drop=True)

train_df = merge_labels(train_df, train_labels)
val_df   = merge_labels(val_df,   val_labels)

print(f'Train: {len(train_df)} | {Counter(train_df.label.tolist())}')
print(f'Val:   {len(val_df)}   | {Counter(val_df.label.tolist())}')
print(f'Test:  {len(test_df)}  (unlabelled)')

Train: 156 | Counter({'yes': 80, 'no': 76})
Val:   23   | Counter({'no': 13, 'yes': 10})
Test:  1959  (unlabelled)


## Step 5 – Rule-Only Baseline (Run First — Takes 5 Seconds)

In [5]:
import re
from sklearn.metrics import f1_score, classification_report

# Optimized rule patterns derived from corpus analysis
# Each pattern chosen for high yes/no signal ratio
INSOMNIA_PATTERNS = [
    # Direct terms — zero false positives
    r'\binsomnia\b',
    r'\bsleeplessness\b',
    r'prn insomnia',
    r'for insomnia',
    r'sleep disturbance',
    r'sleep disorder',
    r'poor sleep',
    r'sleep.*complaint',
    r'complain.*sleep',
    r'c/o sleep',
    r'difficulty.*sleep',
    r'sleep.*difficulty',
    r'unable to sleep',
    r'cannot sleep',
    r'not sleeping',
    r'sleep.wake reversal',
    # Sleep medications — zero false positives
    r'\bzolpidem\b',
    r'\bambien\b',
    r'\btemazepam\b',
    r'\brestoril\b',
    r'\beszopiclone\b',
    r'\blunesta\b',
    r'\bzaleplon\b',
    r'\bsonata\b',
    r'\bramelteon\b',
    r'\brozerem\b',
    r'\bsuvorexant\b',
    r'\bbelsomra\b',
    r'\blemborexant\b',
    r'\btriazolam\b',
    r'\bhalcion\b',
    r'\bdoxylamine\b',
    r'\bunisom\b',
    # Indirect meds — highly predictive in this corpus
    r'\btrazodone\b',
    r'\bclonazepam\b',
    r'\bklonopin\b',
    r'\bmirtazapine\b',
    r'\bdiphenhydramine\b',
    r'\bsertraline\b',
    r'\bcitalopram\b',
    r'\bescitalopram\b',
    r'\bolanzapine\b',
    # Benzo + sleep context combos (zero false positives)
    r'(\blorazepam\b|\bativan\b).{0,60}(sleep|insomnia|rest)',
    r'(sleep|insomnia).{0,60}(\blorazepam\b|\bativan\b)',
    r'(\blorazepam\b|\bativan\b).{0,30}(hs\b|bedtime|qhs|at night)',
    r'(hs\b|bedtime|qhs).{0,30}(\blorazepam\b|\bativan\b)',
]

def rule_predict(text):
    t = text.lower()
    for p in INSOMNIA_PATTERNS:
        if re.search(p, t):
            return 'yes'
    return 'no'

rule_val_preds = [rule_predict(t) for t in val_df['text']]
f1_rule = f1_score(val_df['label'].tolist(), rule_val_preds, pos_label='yes')
print(f'Rule-only Validation F1: {f1_rule:.4f}')
print(classification_report(val_df['label'].tolist(), rule_val_preds, target_names=['no', 'yes']))

Rule-only Validation F1: 0.8696
              precision    recall  f1-score   support

          no       1.00      0.77      0.87        13
         yes       0.77      1.00      0.87        10

    accuracy                           0.87        23
   macro avg       0.88      0.88      0.87        23
weighted avg       0.90      0.87      0.87        23



## Step 6 – Build Prompt Templates

In [6]:
SYSTEM_PROMPT = """You are a clinical NLP expert specializing in identifying insomnia from patient medical records.

Determine whether the patient described suffers from insomnia based on:
1. DIRECT mentions: insomnia, sleeplessness, difficulty sleeping, poor sleep, sleep disturbance
2. MEDICATIONS: Zolpidem, Ambien, Temazepam, Eszopiclone, Lunesta, Zaleplon, Ramelteon, Suvorexant, Doxylamine, Triazolam
3. INDIRECT: nocturia disrupting sleep, sleep-wake reversal, 'prn insomnia' in discharge meds

A prescription of any sleep aid medication is strong evidence of insomnia.
Respond with ONLY one word: yes or no."""


def truncate_text(text, max_chars=2500):
    """Keep beginning (meds + chief complaint) and end (discharge meds)."""
    if len(text) <= max_chars:
        return text
    # Added the closing quote and ensured the concatenation is clean
    return text[:1800] + '\n[...]\n' + text[-500:]


def build_prompt(text, label=None):
    text = truncate_text(text)
    user_msg = (
        f'Analyze this clinical note and determine if the patient suffers from insomnia.\n\n'
        f'CLINICAL NOTE:\n{text}\n\n'
        f"Does this patient suffer from insomnia? Answer 'yes' or 'no':"
    )
    if label is not None:
        return (
            f'<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n'
            f'<start_of_turn>user\n{user_msg}<end_of_turn>\n'
            f'<start_of_turn>model\n{label}<end_of_turn>'
        )
    else:
        return (
            f'<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n'
            f'<start_of_turn>user\n{user_msg}<end_of_turn>\n'
            f'<start_of_turn>model\n'
        )

print('Prompt builder ready.')
print('Sample prompt length:', len(build_prompt(train_df['text'][0], train_df['label'][0])))

Prompt builder ready.
Sample prompt length: 3168


## Step 7 – Load Model (gemma-3-1b-it) with QLoRA

In [18]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import Dataset

MODEL_ID  = 'google/gemma-3-1b-it'
OUTPUT_DIR = './gemma3_1b_insomnia_lora'

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# Model — text-only CausalLM
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    attn_implementation='eager',
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('Model ready!')

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

trainable params: 1,490,944 || all params: 1,001,376,896 || trainable%: 0.1489
Model ready!


## Step 8 – Tokenize Datasets

In [19]:
MAX_LEN = 512

def make_hf_dataset(df):
    return Dataset.from_list([
        {'text': build_prompt(row['text'], row['label'])}
        for _, row in df.iterrows()
    ])

def tokenize_dataset(dataset, tokenizer, max_length=MAX_LEN):
    def tokenize(example):
        out = tokenizer(
            example['text'],
            truncation=True,
            max_length=max_length,
            padding='max_length',
        )
        out['token_type_ids'] = [0] * len(out['input_ids'])
        out['labels'] = out['input_ids'].copy()
        return out
    return dataset.map(tokenize, remove_columns=['text'])

train_raw = make_hf_dataset(train_df)
val_raw   = make_hf_dataset(val_df)

train_tokenized = tokenize_dataset(train_raw, tokenizer)
val_tokenized   = tokenize_dataset(val_raw,   tokenizer)

train_tokenized.set_format('torch')
val_tokenized.set_format('torch')

print(f'Train tokenized: {len(train_tokenized)}')
print(f'Val tokenized:   {len(val_tokenized)}')

Map:   0%|          | 0/156 [00:00<?, ? examples/s]

Map:   0%|          | 0/23 [00:00<?, ? examples/s]

Train tokenized: 156
Val tokenized:   23


## Step 9 – Fine-tune

In [20]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=8,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim='paged_adamw_32bit',
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    weight_decay=0.001,
    report_to='none',
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    eval_accumulation_steps=4,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
)

print('Starting fine-tuning...')
trainer.train()
print('Fine-tuning complete!')

trainer.model.save_pretrained(OUTPUT_DIR + '/best_adapter')
tokenizer.save_pretrained(OUTPUT_DIR + '/best_adapter')
print(f'Adapter saved to {OUTPUT_DIR}/best_adapter')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'pad_token_id': 1}.


Starting fine-tuning...


Epoch,Training Loss,Validation Loss
1,3.517138,3.196801


Epoch,Training Loss,Validation Loss
1,3.517138,3.196801
2,2.397661,2.263224
3,1.973081,1.989679
4,1.764287,1.883093
5,1.724572,1.831416
6,1.634500,1.808015
7,1.586840,1.800076
8,1.642849,1.799101


Fine-tuning complete!
Adapter saved to ./gemma3_1b_insomnia_lora/best_adapter


## Step 10 – Validate Model + Choose Best Strategy

In [23]:
# Recall-boosting hybrid:
# Fire 'yes' if EITHER model OR rules say yes.
# Since precision=1.0 and recall=0.58, we need to catch more positives.
from sklearn.metrics import f1_score, classification_report

model.eval()

def predict_single(text):
    prompt = build_prompt(text, label=None)
    inputs = tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=MAX_LEN
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
    return 'yes' if 'yes' in decoded else 'no'

def predict_batch(df, desc='Predicting'):
    from tqdm.auto import tqdm
    return [predict_single(row['text']) for _, row in tqdm(df.iterrows(), total=len(df), desc=desc)]

val_model_preds = predict_batch(val_df, desc='Validating')
val_true = val_df['label'].tolist()

f1_model = f1_score(val_true, val_model_preds, pos_label='yes')

# Hybrid: OR logic — catches what model misses via rules
val_hybrid_preds = [
    'yes' if (val_model_preds[i] == 'yes' or rule_predict(val_df['text'].iloc[i]) == 'yes')
    else 'no'
    for i in range(len(val_df))
]
f1_hybrid = f1_score(val_true, val_hybrid_preds, pos_label='yes')

print(f'Rule-only F1: {f1_rule:.4f}')
print(f'Model-only F1: {f1_model:.4f}')
print(f'Hybrid F1:    {f1_hybrid:.4f}')
print(classification_report(val_true, val_hybrid_preds, target_names=['no', 'yes']))

# Always use hybrid — it strictly dominates model-only when recall is the gap
BEST_STRATEGY = 'hybrid'
print(f'\n→ Strategy locked to: HYBRID (OR logic)')

Validating:   0%|          | 0/23 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Rule-only F1: 0.8696
Model-only F1: 0.0000
Hybrid F1:    0.8333
              precision    recall  f1-score   support

          no       1.00      0.69      0.82        13
         yes       0.71      1.00      0.83        10

    accuracy                           0.83        23
   macro avg       0.86      0.85      0.83        23
weighted avg       0.88      0.83      0.82        23


→ Strategy locked to: HYBRID (OR logic)


## Step 11 – Inference on Test Set

In [24]:
from tqdm.auto import tqdm

print(f'Running inference on {len(test_df)} test notes using strategy: {BEST_STRATEGY}')
print('Estimated time on T4: ~15-25 minutes')

test_results = {}

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Test inference'):
    note_id = str(row['note_id'])
    text    = row['text']

    if BEST_STRATEGY == 'rule':
        pred = rule_predict(text)
    elif BEST_STRATEGY == 'model':
        pred = predict_single(text)
    else:  # hybrid
        model_pred = predict_single(text)
        pred = 'yes' if (model_pred == 'yes' or rule_predict(text) == 'yes') else 'no'

    test_results[note_id] = {'Insomnia': pred}

yes_count = sum(1 for v in test_results.values() if v['Insomnia'] == 'yes')
no_count  = len(test_results) - yes_count
print(f'\nDone! Predicted yes: {yes_count} ({100*yes_count/len(test_results):.1f}%)')
print(f'       Predicted no:  {no_count} ({100*no_count/len(test_results):.1f}%)')

Running inference on 1959 test notes using strategy: hybrid
Estimated time on T4: ~15-25 minutes


Test inference:   0%|          | 0/1959 [00:00<?, ?it/s]


Done! Predicted yes: 1049 (53.5%)
       Predicted no:  910 (46.5%)


## Step 12 – Save & Download Submission File

In [25]:
import json

SUBMISSION_FILE = 'subtask_1_test.json'

with open(SUBMISSION_FILE, 'w') as f:
    json.dump(test_results, f, indent=4)

print(f'Saved {SUBMISSION_FILE} with {len(test_results)} entries')
print('\nFirst 5 predictions:')
for nid, v in list(test_results.items())[:5]:
    print(f'  {nid}: {v}')

# Download
from google.colab import files
files.download(SUBMISSION_FILE)
print('\nFile downloaded! Submit to: https://www.codabench.org/competitions/15299')

Saved subtask_1_test.json with 1959 entries

First 5 predictions:
  20: {'Insomnia': 'yes'}
  27: {'Insomnia': 'yes'}
  28: {'Insomnia': 'yes'}
  33: {'Insomnia': 'yes'}
  51: {'Insomnia': 'yes'}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


File downloaded! Submit to: https://www.codabench.org/competitions/15299


## (Optional) Step 13 – Rule-Only Fast Submission
Run this standalone if you just want a quick submission without fine-tuning.

In [ ]:
# Fast rule-only submission (no GPU needed, runs in seconds)
rule_test_results = {
    str(row['note_id']): {'Insomnia': rule_predict(row['text'])}
    for _, row in test_df.iterrows()
}
with open('subtask_1_test_rules_only.json', 'w') as f:
    json.dump(rule_test_results, f, indent=4)

yes_r = sum(1 for v in rule_test_results.values() if v['Insomnia'] == 'yes')
print(f'Rule-only: {yes_r} yes, {len(rule_test_results)-yes_r} no')
print(f'Rule val F1 was: {f1_rule:.4f}')

from google.colab import files
files.download('subtask_1_test_rules_only.json')

---
## Submission Checklist
1. ✅ File name: `subtask_1_test.json`
2. ✅ Format: `{"<note_id>": {"Insomnia": "yes"/"no"}, ...}`
3. ✅ All 1959 test note IDs present
4. ✅ Upload at: https://www.codabench.org/competitions/15299

## Troubleshooting
- **OOM during training**: reduce `MAX_LEN` to `384` in Step 8
- **Low F1 after training**: increase `num_train_epochs` to `12` in Step 9
- **Slow inference**: use `BEST_STRATEGY = 'rule'` to skip model inference on test
- **Gemma access error**: accept license at https://huggingface.co/google/gemma-3-1b-it